# 🧬 01 — Aquisição e Curadoria: Receptor de Adenosina A2A (ChEMBL)
**Projeto:** MultiTarget BioPredict (Triagem Virtual Multiobjetivo $A_{2A}$ × hERG)  
**Fase:** 1 — Aquisição e Curadoria de Dados  
**Autor:** Raphael Pires

---

## 🎯 Contexto Biológico e Farmacológico

O **Receptor de Adenosina $A_{2A}$** (*ADORA2A*) é um receptor acoplado à proteína G heterotrimérica ($G_s$). Quando ativado pelo agonista endógeno (adenosina), estimula a adenilato ciclase, elevando as concentrações intracelulares de AMP cíclico (cAMP).

- **No Sistema Nervoso Central:** Expresso abundantemente no corpo estriado, modula a sinalização dopaminérgica $D_2$. O bloqueio por antagonistas como a **Cafeína** e a **Istradefilina** (aprovada pela FDA para doença de Parkinson) reduz a fadiga e otimiza o controle motor.
- **Na Imuno-oncologia:** Níveis elevados de adenosina no microambiente tumoral suprimem linfócitos T citotóxicos e células NK. Antagonistas de $A_{2A}$ atuam como agentes imunoterapêuticos promissores.

---

## 🔬 Escopo do Pipeline
Este caderno realiza o processo completo de curadoria de dados para o **Modelo 1 (Regressão de Afinidade $pIC_{50}$)**:
1. Extração de bioatividades do alvo `CHEMBL251` (humano) via API REST oficial do ChEMBL.
2. Filtragem de ensaios quantitativos definitivos ($IC_{50}$, relação exata `=`, unidades normalizadas em $\text{nM}$).
3. Higienização química via **RDKit**: remoção de contraíons/sais, neutralização de cargas elétricas e geração de **SMILES canônico isomérico**.
4. Conversão da potência para escala logarítmica: $pIC_{50} = -\log_{10}(IC_{50} \times 10^{-9}) = 9.0 - \log_{10}(IC_{50})$.
5. Agrupamento por estrutura molecular única e resolução de duplicatas experimentais (mediana e descarte de discordâncias $\text{std} > 1.0$).
6. Análise exploratória e mapeamento dos semáforos de potência do PRD.
7. Exportação do dataset curado final para `data/processed/a2a_curated.csv`.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# Adiciona o diretório raiz do projeto ao path para carregar módulos do src/
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.chem_utils import canonicalize_smiles
from src.data_utils import fetch_chembl_a2a_dataset, curate_a2a_dataset, RAW_DIR, PROCESSED_DIR

print(f"Diretório raiz do projeto: {project_root}")
print(f"Diretório de dados brutos: {RAW_DIR}")
print(f"Diretório de dados processados: {PROCESSED_DIR}")

## 1. Aquisição dos Dados Brutos do ChEMBL (Target CHEMBL251)

Consultamos a API REST do ChEMBL pelo identificador de alvo **`CHEMBL251`** (Adenosine receptor A2a humano), recuperando todas as medições de ensaios com tipo padrão $IC_{50}$.  
Para assegurar reprodutibilidade científica e evitar requisições redundantes na rede, o resultado bruto é persistido em cache local em `data/raw/chembl_a2a_raw.csv`.

In [ ]:
df_raw = fetch_chembl_a2a_dataset(dest_dir=RAW_DIR, force=False)
print(f"Total de registros de bioatividades brutas recuperados: {len(df_raw)}")
print("\nColunas disponíveis:")
print(list(df_raw.columns))

# Amostra dos dados brutos
df_raw[['activity_id', 'molecule_chembl_id', 'canonical_smiles', 'standard_type', 'standard_relation', 'standard_value', 'standard_units']].head()

## 2. Inspeção e Filtragem Estrita de Integridade

Em bancos como o ChEMBL, ensaios biológicos incluem medições com relações de desigualdade (ex.: `> 10000 nM`, que apenas indicam inatividade) ou unidades não padronizadas.  
Para garantir um conjunto consistente para regressão supervisionada, aplicamos os seguintes filtros:
- Relação exata: `standard_relation == '='`
- Tipo de ensaio: `standard_type == 'IC50'`
- Unidade padronizada: `standard_units == 'nM'`
- Valores numéricos válidos e positivos (`standard_value > 0`)
- Presença de representação química (`canonical_smiles` não-nulo)

In [ ]:
mask = (
    (df_raw['standard_relation'] == '=') &
    (df_raw['standard_type'] == 'IC50') &
    (df_raw['standard_units'] == 'nM') &
    (df_raw['standard_value'].notna()) &
    (df_raw['canonical_smiles'].notna())
)
df_filtered = df_raw.loc[mask].copy()
df_filtered['standard_value'] = pd.to_numeric(df_filtered['standard_value'], errors='coerce')
df_filtered = df_filtered[df_filtered['standard_value'] > 0].copy()

print(f"Registros brutos: {len(df_raw)}")
print(f"Registros válidos com IC50 quantitativo exato: {len(df_filtered)} ({len(df_filtered)/len(df_raw):.1%})")

## 3. Padronização Química via RDKit

A mesma entidade química pode ser sintetizada e testada sob formas distintas (ex.: sais de cloridrato, tautômeros, formas desprotonadas).  
Utilizamos a função centralizada `canonicalize_smiles` (`src/chem_utils.py`), que executa em cada molécula:
1. **Cleanup**: sanitização e remoção de hidrogênios explícitos desnecessários.
2. **LargestFragmentChooser**: descarte de contraíons e moléculas de solvente (ex.: `CCN.Cl` $\rightarrow$ `CCN`).
3. **Uncharger**: neutralização de cargas simples (ex.: íons carboxilato $\rightarrow$ ácidos carboxílicos).
4. **Isomeric SMILES canônico**: garante que representações equivalentes resultem exatamente na mesma string.

In [ ]:
print("Executando padronização química via RDKit...")
df_filtered['clean_smiles'] = df_filtered['canonical_smiles'].astype(str).apply(canonicalize_smiles)

# Descarte de eventuais falhas de conversão ou estruturas inválidas
df_clean = df_filtered[df_filtered['clean_smiles'].notna()].copy()
print(f"Estruturas padronizadas com sucesso pelo RDKit: {len(df_clean)} ({len(df_clean)/len(df_filtered):.1%})")

## 4. Cálculo de Potência ($pIC_{50}$) e Deduplicação Experimental

Em química medicinal e farmacologia quantitativa, a potência é expressa na escala logarítmica negativa:
$$pIC_{50} = -\log_{10}(IC_{50} \times 10^{-9}) = 9.0 - \log_{10}(IC_{50})$$

### Resolução de Duplicatas:
Muitas moléculas foram sintetizadas e ensaiadas em publicações independentes ao longo de anos:
- Agrupamos por `clean_smiles`.
- Calculamos a **mediana** do $pIC_{50}$ para mitigar variações de bancada.
- Calculamos o **desvio padrão** ($std$) entre medições independentes.
- **Controle de Qualidade:** Descartamos moléculas com $std > 1.0$ (discrepância superior a uma ordem de grandeza de concentração entre ensaios, indicando incompatibilidade experimental severa).

In [ ]:
# Aplica a função de curadoria consolidada do src/data_utils.py
df_curated = curate_a2a_dataset(df_raw, max_std=1.0)

print(f"Total de moléculas únicas curadas: {len(df_curated)}")
print(f"Compostos com medições laboratoriais múltiplas: {len(df_curated[df_curated['num_records'] > 1])}")
df_curated.head(10)

## 5. Análise Exploratória e Semáforos do PRD

De acordo com o **Product Requirements Document (PRD)** do MultiTarget BioPredict, o modelo preditivo de $A_{2A}$ atuará com o seguinte semáforo independente de eficácia:
- 🟢 **Alta Potência:** $pIC_{50} \ge 6.5$ (afinidade nanomolar ou subnanomolar, compostos altamente promissores)
- 🟡 **Potência Moderada:** $5.0 \le pIC_{50} < 6.5$ (afinidade micromolar, candidatos a otimização de fragmentos)
- 🔴 **Baixa/Nenhuma Potência:** $pIC_{50} < 5.0$ (fracamente ativos ou inativos)

In [ ]:
# Estatísticas descritivas da distribuição contínua
print("Resumo estatístico do pIC50 curado:")
display(df_curated['pIC50'].describe().to_frame().T)

# Categorização segundo o semáforo do PRD
def get_a2a_semaphore(p):
    if p >= 6.5:
        return "🟢 Alta Potência (>= 6.5)"
    elif p >= 5.0:
        return "🟡 Potência Moderada (5.0 - 6.5)"
    else:
        return "🔴 Baixa/Inativo (< 5.0)"

df_curated['semaphore'] = df_curated['pIC50'].apply(get_a2a_semaphore)
sem_counts = df_curated['semaphore'].value_counts()
sem_props = df_curated['semaphore'].value_counts(normalize=True).mul(100).round(1)

summary_sem = pd.DataFrame({'Total': sem_counts, 'Percentual (%)': sem_props})
print("\nDistribuição das moléculas por Semáforo de Eficácia:")
display(summary_sem)

## 6. Exportação do Dataset Curado

Gravamos a tabela curada definitiva em `data/processed/a2a_curated.csv` pronta para a **Fase 2 (Featurização e Engenharia de Atributos)**.

In [ ]:
out_path = PROCESSED_DIR / "a2a_curated.csv"
export_cols = ["molecule_chembl_id", "canonical_smiles", "pIC50", "num_records", "pIC50_std"]
df_curated[export_cols].to_csv(out_path, index=False)

print(f"Dataset curado exportado com sucesso: {out_path}")
print(f"Dimensões finais: {df_curated[export_cols].shape}")
print(f"Colunas: {export_cols}")